In [ ]:
# https://www.kaggle.com/datasets/datasetengineer/cybersecurity-threat-and-awareness-program-dataset

In [253]:
import getpass
import os
from langchain.chat_models import init_chat_model
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

In [1]:
if not os.environ.get("GROQ_API_KEY"):
  os.environ["GROQ_API_KEY"] = getpass.getpass("Enter API key for Groq: ")

Enter API key for Groq:  ········


In [11]:
def callLLM(messages):
    llm = init_chat_model("llama3-8b-8192", model_provider="groq")
    return llm.invoke(messages)

In [26]:
logs_ds = pd.read_csv("/Users/puneetrathore/Documents/Puneet/input/CTDAPD Dataset.csv")
logs_ds["log_id"] = list(range(len(logs_ds)))

In [3]:
all_cols = ["log_id"]+list(logs_ds.columns[:-1])

input_cols = ['log_id','Date', 'Source_IP', 'Destination_IP', 'Source_Port',
       'Destination_Port', 'Protocol_Type', 'Flow_Duration', 'Packet_Size',
       'Flow_Bytes_per_s', 'Flow_Packets_per_s', 'Total_Forward_Packets',
       'Total_Backward_Packets', 'Packet_Length_Mean_Forward',
       'Packet_Length_Mean_Backward', 'IAT_Forward', 'IAT_Backward',
       'Active_Duration', 'Idle_Duration','CPU_Utilization', 'Memory_Utilization', 'System_Patch_Status',
       'Normalized_Packet_Flow','Anomaly_Score']

In [4]:
logs_ds.groupby("Label")["Label"].count()

Label
Attack     8179
Normal    46589
Name: Label, dtype: int64

### Select stratified sampled data for testing LLM

In [131]:
norm_df = logs_ds[logs_ds["Label"] == "Normal"]
attack_df = logs_ds[logs_ds["Label"] == "Attack"]

sampledDF = pd.concat([norm_df.head(20),attack_df.head(2)], ignore_index=True).sample(frac=1).reset_index(drop=True)

len(norm_df), len(sampledDF)

(46589, 22)

In [132]:
sampledDF.groupby("Label")["Label"].count()

Label
Attack     2
Normal    20
Name: Label, dtype: int64

In [133]:
logs_data = sampledDF[input_cols].to_dict(orient='index')

### Test LLM as Agents

In [266]:
features_desc = """
Source_IP: IP address of the originating device involved in the event (e.g., 192.168.1.1).
Destination_IP: IP address of the target device involved in the event (e.g., 10.0.0.5).
Source_Port: Port number on the originating device (e.g., 443).
Destination_Port: Port number on the target device (e.g., 80).
Protocol_Type: The protocol used for the communication, such as TCP, UDP, ICMP.
Flow_Duration: Duration of the network flow in milliseconds.
Packet_Size: The size of the packet in bytes.
Flow_Bytes/s: The number of bytes transmitted per second during the flow.
Flow_Packets/s: The number of packets transmitted per second during the flow.
Total_Forward_Packets: Total number of packets sent in the forward direction.
Total_Backward_Packets: Total number of packets sent in the reverse direction.
Packet_Length_Mean_Forward: Average forward packet length for the flow.
Packet_Length_Mean_Backward: Average backward packet length for the flow.
IAT_Forward: Inter-arrival time for packets in the forward direction.
IAT_Backward: Inter-arrival time for packets in the reverse direction.
Active_Duration: Duration of active time for the connection.
Idle_Duration: Duration of idle time for the connection.
CPU_Utilization: CPU usage percentage during the event.
Memory_Utilization: Memory usage percentage during the event.
System_Patch_Status:  Indicates whether the system is patched (Up-to-date, Outdated).
Normalized_Packet_Flow: normalized number of packets transmitted per second during the flow.
"""

In [154]:
####
## Convert dict to text logs, as expected input
###

messages = """From these 22 activity logs, identify and return only the log_id for top 2 activities that looks anomolous or suspicious.
logs = {l}
""".format(l=logs_data, d=features_desc)

In [155]:
# messages

### Call Agent

In [156]:
resp = callLLM(messages)

In [157]:
print(resp.content)

After analyzing the activity logs, I have identified the top 2 logs that look anomalous or suspicious. Here are the log IDs:

1. **Log ID: 14** (Date: 2018-01-01 14:00:00)
	* Anomaly Score: 67.70010184711808
	* Flow Duration: 153
	* Packet Size: 467
	* Flow Bytes per s: 5.139543004683451
	* Flow Packets per s: 2.6729032123566876
	* Total Forward Packets: 17
	* Total Backward Packets: 7
	* System Patch Status: Outdated
	* Normalized Packet Flow: 0.1568627450980392

2. **Log ID: 18** (Date: 2018-01-01 18:00:00)
	* Anomaly Score: 46.9355920244698
	* Flow Duration: 27
	* Packet Size: 629
	* Flow Bytes per s: 8.46108462689364
	* Flow Packets per s: 3.0275854944572123
	* Total Forward Packets: 17
	* Total Backward Packets: 14
	* System Patch Status: Outdated
	* Normalized Packet Flow: 1.148148148148148

These logs have high Anomaly Scores, indicating that they may be unusual or suspicious. The logs have longer flow durations, larger packet sizes, and higher flow rates compared to other logs.

In [158]:
sampledDF[sampledDF["Label"] == "Attack"][["log_id","CPU_Utilization","Anomaly_Score"]]

,log_id,CPU_Utilization,CPU_Utilization,Anomaly_Score
1,6,30.275541,30.275541,34.972284
8,1,31.986167,31.986167,23.612143


In [159]:
sampledDF[sampledDF["log_id"].isin([14,18])][["log_id","Anomaly_Score","Label"]]

,log_id,Anomaly_Score,Label
7,14,67.700102,Normal
18,18,46.935592,Normal


In [148]:
sampledDF[["log_id","Label"]]

,log_id,Label
0,0,Normal
1,6,Attack
2,2,Normal
3,12,Normal
4,15,Normal
5,8,Normal
6,16,Normal
7,14,Normal
8,1,Attack
9,24,Normal


### Apply time series anomaly detection using only LLM

In [207]:
from sklearn.model_selection import train_test_split
from collections import Counter
from sklearn.ensemble import IsolationForest
import numpy as np
from sklearn.metrics import precision_recall_fscore_support as score
import matplotlib.pyplot as plt

In [254]:
logs_ds = pd.read_csv("/Users/puneetrathore/Documents/Puneet/input/CTDAPD Dataset.csv")
logs_ds["log_id"] = list(range(len(logs_ds)))

In [288]:
logs_ds

,Date,Source_IP,Destination_IP,Source_Port,Destination_Port,Protocol_Type,Flow_Duration,Packet_Size,Flow_Bytes_per_s,Flow_Packets_per_s,...,Attack_Severity,Botnet_Family,Malware_Type,CPU_Utilization,Memory_Utilization,System_Patch_Status,Label,Normalized_Packet_Flow,Anomaly_Severity_Index,log_id
0,2018-01-01 00:00:00,192.168.1.1,8.8.8.8,57708,443,TCP,15,500,6.375467,5.414037,...,Low,NaN,NaN,16.375491,60.918547,Outdated,Normal,2.000000,73.168788,0
1,2018-01-01 01:00:00,172.16.0.1,4.4.4.4,29980,8080,TCP,390,472,6.694334,2.497333,...,Low,NaN,NaN,31.986167,7.718833,Outdated,Attack,0.102564,16.059524,1
2,2018-01-01 02:00:00,10.0.0.1,8.8.8.8,31923,443,TCP,120,129,7.948582,3.607725,...,Medium,NaN,NaN,1.919325,51.726672,Up-to-date,Normal,0.233333,99.333361,2
3,2018-01-01 03:00:00,192.168.1.1,8.8.8.8,10525,80,TCP,3,500,7.326526,4.144272,...,Low,NaN,NaN,21.078560,20.463289,Up-to-date,Normal,9.666667,46.645165,3
4,2018-01-01 04:00:00,192.168.1.1,1.1.1.1,31755,80,UDP,7,458,7.080846,2.667007,...,Medium,NaN,NaN,24.940597,43.652572,Up-to-date,Normal,3.571429,31.648643,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54763,2024-03-31 19:00:00,192.168.1.1,8.8.8.8,55784,80,ICMP,16,352,7.798953,2.326618,...,Low,NaN,NaN,35.654680,57.471818,Up-to-date,Normal,2.562500,37.694837,54763
54764,2024-03-31 20:00:00,192.168.1.1,8.8.8.8,51055,80,TCP,0,649,5.104806,3.695383,...,Medium,NaN,NaN,22.459519,34.834888,Up-to-date,Normal,inf,37.940200,54764
54765,2024-03-31 21:00:00,192.168.1.1,8.8.8.8,40203,443,TCP,200,848,13.540904,2.202821,...,High,NaN,NaN,27.772790,10.906432,Outdated,Normal,0.145000,49.541831,54765
54766,2024-03-31 22:00:00,192.168.1.1,8.8.8.8,46248,8080,TCP,154,474,11.372500,2.262444,...,Low,NaN,Spyware,43.175241,25.847987,Up-to-date,Normal,0.253247,43.470523,54766


In [255]:
input_cols = ['Date', 'Source_IP', 'Destination_IP', 'Source_Port',
       'Destination_Port', 'Protocol_Type', 'Flow_Duration', 'Packet_Size',
       'Flow_Bytes_per_s', 'Flow_Packets_per_s', 'Total_Forward_Packets',
       'Total_Backward_Packets', 'Packet_Length_Mean_Forward',
       'Packet_Length_Mean_Backward', 'IAT_Forward', 'IAT_Backward',
       'Active_Duration', 'Idle_Duration',
       'CPU_Utilization', 'Memory_Utilization', 'System_Patch_Status',
       'Normalized_Packet_Flow']

In [256]:
oneMonthDF = logs_ds[(logs_ds["Date"] >= "2024-03-01 00:00:00") & (logs_ds["Date"] <= "2024-03-31 00:00:00")]

# Select numeric columns
numeric_df = oneMonthDF.select_dtypes(include=np.number)

# Round to 2 decimal places
oneMonthDF[numeric_df.columns] = numeric_df.round(2)

oneMonthDF.groupby("Label")["Label"].count()

Label
Attack    129
Normal    592
Name: Label, dtype: int64

In [276]:
oneMonthDF["date"] = pd.to_datetime(oneMonthDF["Date"]).dt.date
oneMonthDF[oneMonthDF["Attack_Vector"] == "DDoS"]["Date"].unique()

array(['2024-03-01 03:00:00', '2024-03-02 02:00:00',
       '2024-03-02 09:00:00', '2024-03-03 05:00:00',
       '2024-03-03 07:00:00', '2024-03-04 06:00:00',
       '2024-03-04 09:00:00', '2024-03-04 20:00:00',
       '2024-03-05 06:00:00', '2024-03-05 10:00:00',
       '2024-03-05 14:00:00', '2024-03-05 21:00:00',
       '2024-03-05 23:00:00', '2024-03-06 00:00:00',
       '2024-03-08 00:00:00', '2024-03-08 09:00:00',
       '2024-03-08 23:00:00', '2024-03-09 14:00:00',
       '2024-03-10 04:00:00', '2024-03-10 05:00:00',
       '2024-03-10 22:00:00', '2024-03-11 12:00:00',
       '2024-03-11 22:00:00', '2024-03-12 01:00:00',
       '2024-03-12 18:00:00', '2024-03-13 03:00:00',
       '2024-03-13 06:00:00', '2024-03-13 10:00:00',
       '2024-03-13 13:00:00', '2024-03-13 15:00:00',
       '2024-03-14 01:00:00', '2024-03-14 05:00:00',
       '2024-03-14 11:00:00', '2024-03-14 15:00:00',
       '2024-03-15 03:00:00', '2024-03-15 11:00:00',
       '2024-03-16 02:00:00', '2024-03-16 08:0

In [258]:
T_DF = oneMonthDF[input_cols].set_index("Date").T

In [259]:
T_DF.head(100)

Date,2024-03-01 00:00:00,2024-03-01 01:00:00,2024-03-01 02:00:00,2024-03-01 03:00:00,2024-03-01 04:00:00,2024-03-01 05:00:00,2024-03-01 06:00:00,2024-03-01 07:00:00,2024-03-01 08:00:00,2024-03-01 09:00:00,...,2024-03-30 15:00:00,2024-03-30 16:00:00,2024-03-30 17:00:00,2024-03-30 18:00:00,2024-03-30 19:00:00,2024-03-30 20:00:00,2024-03-30 21:00:00,2024-03-30 22:00:00,2024-03-30 23:00:00,2024-03-31 00:00:00
Source_IP,192.168.1.1,192.168.1.1,172.16.0.1,10.0.0.1,192.168.1.1,192.168.1.1,192.168.1.1,192.168.1.1,10.0.0.1,192.168.1.1,...,192.168.1.1,192.168.1.1,192.168.1.1,192.168.1.1,192.168.1.1,10.0.0.1,192.168.1.1,192.168.1.1,192.168.1.1,192.168.1.1
Destination_IP,8.8.8.8,8.8.8.8,4.4.4.4,8.8.8.8,8.8.8.8,8.8.8.8,4.4.4.4,1.1.1.1,4.4.4.4,4.4.4.4,...,8.8.8.8,8.8.8.8,4.4.4.4,8.8.8.8,8.8.8.8,8.8.8.8,8.8.8.8,8.8.8.8,4.4.4.4,4.4.4.4
Source_Port,55043,59818,10834,33918,33233,36783,5221,59121,9780,39223,...,46671,60757,12345,51973,14368,33426,28655,34574,24558,21851
Destination_Port,80,443,80,80,443,443,443,80,8080,443,...,22,443,80,80,80,80,8080,80,443,443
Protocol_Type,TCP,TCP,UDP,TCP,UDP,TCP,TCP,TCP,UDP,ICMP,...,ICMP,ICMP,TCP,TCP,UDP,ICMP,TCP,UDP,TCP,TCP
Flow_Duration,23,23,4,66,67,108,43,80,88,91,...,129,55,82,8,157,123,113,203,147,570
Packet_Size,757,485,258,493,699,413,460,458,825,319,...,613,508,584,631,558,701,568,502,701,765
Flow_Bytes_per_s,7.95,15.28,4.66,5.76,4.82,9.03,17.85,5.9,3.29,3.93,...,17.24,10.98,7.14,7.85,7.78,3.98,10.19,3.74,7.13,14.84
Flow_Packets_per_s,3.82,1.97,1.82,2.46,2.53,3.09,1.5,2.51,3.01,2.94,...,2.19,1.99,1.87,1.85,4.02,3.78,2.98,1.75,3.94,2.14
Total_Forward_Packets,22,26,20,21,20,15,17,28,29,16,...,16,21,19,22,27,20,17,16,14,25


In [292]:
import google.generativeai as genai
import os

API_KEY = '' # Replace with your actual API key

# Configure the Gemini API with your API key
genai.configure(api_key=API_KEY)

# Initialize the Generative Model (using 'gemini-pro' as an example)
# You can choose other models if available, e.g., 'gemini-1.5-flash', 'gemini-1.5-pro'
text_model = genai.GenerativeModel(model_name = 'gemini-2.0-flash', tools = 'code_execution')

print("Gemini model initialized successfully!")

# You can add generation configurations and safety settings
generation_config = {
    "temperature": 0.1,  # Controls randomness. Higher values mean more random outputs.
    "top_p": 1,          # Nucleus sampling.
    "top_k": 1,          # Top-k sampling.
    "max_output_tokens": 500, # Maximum number of tokens in the response.
}

safety_settings = [
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
]


def send_message(message):
    try:
        response = text_model.generate_content(message)
        # The response.text will contain the Python code generated by Gemini
        print("Generated Python Code:")
        return response.text
    except Exception as e:
        print(f"An error occurred during API call: {e}")
        print("Please ensure your API key is correct and you have network connectivity.")


Gemini model initialized successfully!


#### Checking with data as-it-is.

In [293]:
T_DF_str = T_DF.to_dict(orient='list')
len(T_DF)

21

In [294]:
T_DF.index

Index(['Source_IP', 'Destination_IP', 'Source_Port', 'Destination_Port',
       'Protocol_Type', 'Flow_Duration', 'Packet_Size', 'Flow_Bytes_per_s',
       'Flow_Packets_per_s', 'Total_Forward_Packets', 'Total_Backward_Packets',
       'Packet_Length_Mean_Forward', 'Packet_Length_Mean_Backward',
       'IAT_Forward', 'IAT_Backward', 'Active_Duration', 'Idle_Duration',
       'CPU_Utilization', 'Memory_Utilization', 'System_Patch_Status',
       'Normalized_Packet_Flow'],
      dtype='object')

In [295]:
input_prompt = """You are a network traffic analyzer.
Below are the network traffic logs arranged such that each key represents the hour and values contain the network 
activites in the same order, as explained here: {d}
Analyze below logs for 721 hours, in a duration of 30 days to identify the hours where the activity looks anomalous or suspicious, 
represnting an unusual pattern.
network traffic logs = {l}
""".format(l=T_DF_str, d=features_desc)

G_resp = send_message(input_prompt)

Generated Python Code:


In [291]:
print(input_prompt)

You are a network traffic analyzer.
Below are the network traffic logs arranged such that each key represents the hour and values contain the network 
activites in the same order, as explained here: 
Source_IP: IP address of the originating device involved in the event (e.g., 192.168.1.1).
Destination_IP: IP address of the target device involved in the event (e.g., 10.0.0.5).
Source_Port: Port number on the originating device (e.g., 443).
Destination_Port: Port number on the target device (e.g., 80).
Protocol_Type: The protocol used for the communication, such as TCP, UDP, ICMP.
Flow_Duration: Duration of the network flow in milliseconds.
Packet_Size: The size of the packet in bytes.
Flow_Bytes/s: The number of bytes transmitted per second during the flow.
Flow_Packets/s: The number of packets transmitted per second during the flow.
Total_Forward_Packets: Total number of packets sent in the forward direction.
Total_Backward_Packets: Total number of packets sent in the reverse direction

In [296]:
print(G_resp)

Okay, I will analyze the provided network traffic logs spanning 721 hours (30 days) to identify potentially anomalous or suspicious activity. I will focus on unusual patterns and report the hours where such patterns are observed.  Since I cannot analyze all 721 hours, I will focus on the potentially most suspicious hours in the given data.

I'll be looking for:

*   **Unusual Protocol Types:** Spikes in ICMP or unusual UDP traffic.
*   **High Packet/Byte Rates:**  Sudden increases in `Flow_Bytes/s` and `Flow_Packets/s`.
*   **Long Flow Durations:** Abnormally long `Flow_Duration` values.
*   **Patch Status:** Hours where systems with `Outdated` patches are experiencing high traffic or suspicious activity.
*   **High CPU/Memory Utilization:**  Correlations between high utilization and suspicious traffic patterns.
*   **Unusual Port Combinations:** Traffic on unexpected port combinations.
*   **Normalized Packet Flow:** Sudden spikes.
*   **Idle and Active Duration:** Look for very large

#### Review output

In [285]:
data_ddos_ts = ['2024-03-01 03:00:00', '2024-03-02 02:00:00', '2024-03-02 09:00:00', '2024-03-03 05:00:00', '2024-03-03 07:00:00', '2024-03-04 06:00:00', '2024-03-04 09:00:00', '2024-03-04 20:00:00', '2024-03-05 06:00:00', '2024-03-05 10:00:00', '2024-03-05 14:00:00', '2024-03-05 21:00:00', '2024-03-05 23:00:00', '2024-03-06 00:00:00', '2024-03-08 00:00:00', '2024-03-08 09:00:00', '2024-03-08 23:00:00', '2024-03-09 14:00:00', '2024-03-10 04:00:00', '2024-03-10 05:00:00', '2024-03-10 22:00:00', '2024-03-11 12:00:00', '2024-03-11 22:00:00', '2024-03-12 01:00:00', '2024-03-12 18:00:00', '2024-03-13 03:00:00', '2024-03-13 06:00:00', '2024-03-13 10:00:00', '2024-03-13 13:00:00', '2024-03-13 15:00:00', '2024-03-14 01:00:00', '2024-03-14 05:00:00', '2024-03-14 11:00:00', '2024-03-14 15:00:00', '2024-03-15 03:00:00', '2024-03-15 11:00:00', '2024-03-16 02:00:00', '2024-03-16 08:00:00', '2024-03-17 17:00:00', '2024-03-18 06:00:00', '2024-03-18 12:00:00', '2024-03-19 04:00:00', '2024-03-19 13:00:00', '2024-03-20 02:00:00', '2024-03-20 07:00:00', '2024-03-20 08:00:00', '2024-03-20 11:00:00', '2024-03-21 06:00:00', '2024-03-21 10:00:00', '2024-03-21 18:00:00', '2024-03-22 01:00:00', '2024-03-22 03:00:00', '2024-03-22 17:00:00', '2024-03-24 11:00:00', '2024-03-24 17:00:00', '2024-03-25 00:00:00', '2024-03-25 07:00:00', '2024-03-25 20:00:00', '2024-03-26 02:00:00', '2024-03-26 07:00:00', '2024-03-26 23:00:00', '2024-03-27 08:00:00', '2024-03-27 13:00:00', '2024-03-27 20:00:00', '2024-03-27 22:00:00', '2024-03-28 19:00:00', '2024-03-29 16:00:00', '2024-03-30 01:00:00', '2024-03-30 19:00:00', '2024-03-30 22:00:00', '2024-03-31 00:00:00']
len(data_ddos_ts)

71

In [286]:
aai_identified = ['2024-03-01 02:00:00', '2024-03-01 06:00:00', '2024-03-01 09:00:00', '2024-03-01 12:00:00', '2024-03-02 02:00:00', '2024-03-02 04:00:00', '2024-03-02 08:00:00', '2024-03-02 16:00:00', '2024-03-03 08:00:00', '2024-03-03 13:00:00', '2024-03-03 14:00:00', '2024-03-03 17:00:00', '2024-03-03 23:00:00', '2024-03-04 00:00:00', '2024-03-04 02:00:00', '2024-03-04 09:00:00', '2024-03-04 11:00:00', '2024-03-04 12:00:00', '2024-03-04 14:00:00', '2024-03-04 17:00:00', '2024-03-04 23:00:00', '2024-03-05 01:00:00', '2024-03-05 04:00:00', '2024-03-05 05:00:00', '2024-03-05 16:00:00', '2024-03-05 18:00:00', '2024-03-05 21:00:00', '2024-03-05 23:00:00', '2024-03-06 02:00:00', '2024-03-06 04:00:00', '2024-03-06 06:00:00', '2024-03-06 08:00:00', '2024-03-06 10:00:00', '2024-03-06 19:00:00', '2024-03-07 00:00:00', '2024-03-07 05:00:00', '2024-03-07 06:00:00', '2024-03-07 07:00:00', '2024-03-07 12:00:00', '2024-03-07 14:00:00', '2024-03-07 17:00:00', '2024-03-08 00:00:00', '2024-03-08 11:00:00', '2024-03-08 17:00:00', '2024-03-08 18:00:00', '2024-03-08 23:00:00', '2024-03-09 04:00:00', '2024-03-09 05:00:00', '2024-03-09 06:00:00', '2024-03-09 08:00:00', '2024-03-09 09:00:00', '2024-03-09 10:00:00', '2024-03-09 14:00:00', '2024-03-10 05:00:00', '2024-03-10 10:00:00', '2024-03-10 11:00:00', '2024-03-10 13:00:00', '2024-03-10 14:00:00', '2024-03-11 01:00:00', '2024-03-11 05:00:00', '2024-03-11 17:00:00', '2024-03-11 22:00:00', '2024-03-12 00:00:00', '2024-03-12 10:00:00', '2024-03-12 11:00:00', '2024-03-12 12:00:00', '2024-03-12 15:00:00', '2024-03-12 16:00:00', '2024-03-12 19:00:00', '2024-03-12 21:00:00', '2024-03-12 23:00:00', '2024-03-13 00:00:00', '2024-03-13 06:00:00', '2024-03-13 10:00:00', '2024-03-13 13:00:00', '2024-03-14 00:00:00', '2024-03-14 01:00:00', '2024-03-14 06:00:00', '2024-03-14 07:00:00', '2024-03-14 08:00:00', '2024-03-14 17:00:00', '2024-03-14 18:00:00', '2024-03-14 22:00:00', '2024-03-14 23:00:00', '2024-03-15 01:00:00', '2024-03-15 03:00:00', '2024-03-15 08:00:00', '2024-03-15 16:00:00', '2024-03-16 04:00:00', '2024-03-16 07:00:00', '2024-03-16 08:00:00', '2024-03-16 10:00:00', '2024-03-16 16:00:00', '2024-03-16 19:00:00', '2024-03-16 21:00:00', '2024-03-17 01:00:00', '2024-03-17 04:00:00', '2024-03-17 06:00:00', '2024-03-17 10:00:00', '2024-03-17 12:00:00', '2024-03-17 15:00:00', '2024-03-17 20:00:00', '2024-03-17 21:00:00', '2024-03-18 02:00:00', '2024-03-18 12:00:00', '2024-03-19 02:00:00', '2024-03-19 03:00:00', '2024-03-19 04:00:00', '2024-03-19 07:00:00', '2024-03-19 11:00:00', '2024-03-20 10:00:00', '2024-03-20 12:00:00', '2024-03-20 13:00:00', '2024-03-20 21:00:00', '2024-03-21 01:00:00', '2024-03-21 02:00:00', '2024-03-21 03:00:00', '2024-03-21 05:00:00', '2024-03-21 09:00:00', '2024-03-21 10:00:00', '2024-03-21 17:00:00', '2024-03-21 22:00:00', '2024-03-22 00:00:00', '2024-03-22 04:00:00', '2024-03-22 07:00:00', '2024-03-22 08:00:00', '2024-03-22 10:00:00', '2024-03-22 17:00:00', '2024-03-22 22:00:00', '2024-03-23 01:00:00', '2024-03-23 04:00:00', '2024-03-23 07:00:00', '2024-03-24 07:00:00', '2024-03-24 10:00:00', '2024-03-24 12:00:00', '2024-03-25 02:00:00', '2024-03-25 06:00:00', '2024-03-25 09:00:00', '2024-03-25 12:00:00', '2024-03-25 14:00:00', '2024-03-25 17:00:00', '2024-03-25 22:00:00', '2024-03-26 03:00:00', '2024-03-26 05:00:00', '2024-03-26 09:00:00', '2024-03-26 17:00:00', '2024-03-26 18:00:00', '2024-03-27 01:00:00', '2024-03-27 11:00:00', '2024-03-27 14:00:00', '2024-03-27 17:00:00', '2024-03-27 18:00:00', '2024-03-28 01:00:00', '2024-03-28 02:00:00', '2024-03-28 03:00:00', '2024-03-28 04:00:00', '2024-03-28 05:00:00', '2024-03-28 06:00:00', '2024-03-28 08:00:00', '2024-03-28 09:00:00', '2024-03-28 14:00:00', '2024-03-28 15:00:00', '2024-03-28 18:00:00', '2024-03-28 21:00:00', '2024-03-29 00:00:00', '2024-03-29 05:00:00', '2024-03-29 07:00:00', '2024-03-29 08:00:00', '2024-03-29 14:00:00', '2024-03-29 15:00:00', '2024-03-29 21:00:00', '2024-03-29 23:00:00', '2024-03-30 02:00:00', '2024-03-30 03:00:00', '2024-03-30 04:00:00', '2024-03-30 07:00:00', '2024-03-30 08:00:00', '2024-03-30 09:00:00', '2024-03-30 12:00:00', '2024-03-30 13:00:00', '2024-03-30 18:00:00', '2024-03-31 00:00:00']
len(aai_identified)

182

In [287]:
len([t for t in data_ddos_ts if t in aai_identified])

20